## README
### 1) Problem Framing
#### • Objective: Recommend Reddy-style posts to users and maximize the relevance and diversity of the top K posts in the implicit feedback (upvote) scenario.
#### • Data:
#### • Interaction table (user_id, post_id, vote, created_utc), vote>0 is regarded as positive feedback;
#### • Content table (title, selftext, num_comments, num_unique_commentators, ups, upvote_ratio, author, created_utc).
#### • Challenges: Sparse long-tail (most users/authors/posts have very little interaction), cold start (new posts have no history), popularity bias

### 2) Training/Evaluation Protocol 
#### • Model only on the active subset: First, filter "active users/active posts" according to the EDA selection threshold (reduce the evaluation noise caused by only 1-2 interacting users).
#### • Time-based per-user holdout: Sort each user by time, conduct training/validation in the early part and testing in the late part, simulating "predicting the future with the past" to prevent information leakage.
#### • Evaluation indicators: Use Recall@K, nDCG@K (position-weighted, with a greater focus on top-quality)

### 3) Model Selection
#### • Select LightFM (WARP/BPR) :
#### • Natively supports implicit feedback and negative sampling, suitable for upvote data;
#### • Loss function: warp/Warp-kOS (directly optimize the Top-K sort approximation) is preferred, and BPR is used as an alternative baseline;

### 4) Content Feature Engineering
#### • Semantic vectors (SBERT, all-MiniLM-L6-v2, normalization) : Capture posts with similar semantics and related topics to enhance cold start recall; Normalization avoids vector norms amplifying gradients.
#### • Topic Distribution (LDA+TF-IDF) : Provides interpretable topic dimensions to enhance alignment with users' topic preferences; Select 50 to 200 topics (weighing interpretability and computation).
#### • Emotion (TextBlob polarity/subjectivity) : It supplements the intensity of emotions and subjective signals, facilitating the distinction between writing styles such as "debate/teasing/information".
#### • Normalized metadata: log(num_comments/ups/age_hours/word_count/char_count) + MinMax scaling to reduce double tails and dimensional differences.
#### • Author Bucket (Top-K + other) : Categorize Top authors into one-hot, providing a stable prior knowledge of creators while preventing "unknown" from being mixed into the top.

### 5) Training Details (Design Choices)
#### • Training with only positive feedback: It is most stable to have WARP/BPR treat "not observed" as a negative sample.
#### • all_seen blocking: During the recommendation stage, content that users have seen or stepped on is simultaneously blocked to prevent duplicate recommendations.
#### • Hyperparameter selection: Make a small-scale grid for no_components/loss/reg/lr and select the best one in the validation set using P@200 + nDCG@200; Fixed epochs (such as 25-30) ensure reproducibility.



In [6]:
import kagglehub
import os
import pandas as pd
from datasets import load_dataset
from kagglehub import KaggleDatasetAdapter

In [7]:
def download_dataset()->list[str]:
    """
    Download the dataset from Kaggle and return the paths to the files.
    """
    dataset_dir = kagglehub.dataset_download("josephleake/huge-collection-of-reddit-votes")
    paths = []
    for dir_path, _, file_names in os.walk(dataset_dir):
        for file_name in file_names:
            paths.append(os.path.join(dir_path, file_name))
    print(f'File path to votes:\n{paths[0]}')
    print(f'File path to submissions:\n{paths[1]}')
    return paths

def get_dataframe()->tuple[pd.DataFrame]:
    """
    Return a tuple of two pandas.Dataframe: votes and submissions.

    Returns:
        tuple[pd.DataFrame]: a tuple of two dataframes.
    """
    paths = download_dataset()
    votes = pd.read_csv(paths[0], sep='\t')
    submissions = pd.read_csv(paths[1], sep='\t')
    return (votes, submissions)

In [ ]:
import random
from math import log2, ceil
import numpy as np
import pandas as pd
from lightfm import LightFM
from lightfm.data import Dataset
from lightfm.evaluation import precision_at_k, auc_score
from sentence_transformers import SentenceTransformer
from gensim import corpora
from gensim.models import TfidfModel, LdaModel
from textblob import TextBlob
from sklearn.preprocessing import MinMaxScaler
import nltk
from nltk.corpus import stopwords

# ---- Initialize resources ----
random.seed(42)
np.random.seed(42)

nltk.download("stopwords", quiet=True)
try:
    import textblob.download_corpora as tdc  # type: ignore
    tdc.download_all()
except Exception:
    pass

STOP_WORDS = set(stopwords.words("english"))

# ---- Tools/Processing functions ----
def simple_tokenize(text: str):
    text = str(text).lower()
    text = pd.Series(text).str.replace(r"https?://\S+", " ", regex=True).iloc[0]
    text = pd.Series(text).str.replace(r"[^a-z0-9\s]", " ", regex=True).iloc[0]
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 1]
    return tokens

def per_user_holdout(df, test_frac=0.2, val_frac=0.1, time_col='created_utc'):
    train_rows, val_rows, test_rows = [], [], []
    for uid, uv in df.groupby('user_id'):
        uv = uv.sort_values(time_col)
        n = len(uv)
        if n < 2:
            train_rows.append(uv)
            continue
        split_test = int(ceil(n * (1 - test_frac)))
        train_val_part = uv.iloc[:split_test]
        test_part = uv.iloc[split_test:]

        m = len(train_val_part)
        if m < 2:
            train_rows.append(train_val_part)
            val_rows.append(train_val_part)
        else:
            split_val = int(ceil(m * (1 - val_frac)))
            train_rows.append(train_val_part.iloc[:split_val])
            val_rows.append(train_val_part.iloc[split_val:])

        test_rows.append(test_part)

    train_df = pd.concat(train_rows, ignore_index=True)
    val_df = pd.concat(val_rows, ignore_index=True) if val_rows else pd.DataFrame(columns=df.columns)
    test_df = pd.concat(test_rows, ignore_index=True)
    return train_df, val_df, test_df

def get_topic_vector(lda_model, bow, num_topics):
    dist = np.zeros(num_topics, dtype=float)
    for topic_id, prob in lda_model.get_document_topics(bow, minimum_probability=0):
        dist[topic_id] = prob
    return dist

def ndcg_at_k_batch(model, interactions, k=10, item_features=None):
    n_users, n_items = interactions.shape
    results = []
    csr = interactions.tocsr()
    for u in range(n_users):
        true_mask = csr[u].toarray().ravel() > 0
        if not true_mask.any():
            results.append(0.0)
            continue
        scores = model.predict(np.repeat(u, n_items), np.arange(n_items), item_features=item_features)
        topk = np.argsort(-scores)[:k]
        dcg = 0.0
        for rank, iid in enumerate(topk):
            if true_mask[iid]:
                dcg += 1.0 / log2(rank + 2)
        ideal_n = min(k, true_mask.sum())
        idcg = sum(1.0 / log2(i + 2) for i in range(int(ideal_n)))
        results.append(dcg / idcg if idcg > 0 else 0.0)
    return np.array(results)

def get_top_n_recs_for_all_users(model, hfm_inputs, N=10):
    user_map = hfm_inputs["user_map"]
    item_map = hfm_inputs["item_map"]
    item_features = hfm_inputs["item_features"]
    train_inter = hfm_inputs["train_interactions"]

    n_items = len(item_map)
    topn_indices = {}
    for _, uidx in user_map.items():
        scores = model.predict(np.repeat(uidx, n_items), np.arange(n_items), item_features=item_features)
        seen = train_inter.tocsr()[uidx].toarray().ravel() > 0
        scores[seen] = -np.inf
        top_idxs = np.argsort(-scores)[:N]
        topn_indices[uidx] = top_idxs
    return topn_indices

def recall_at_k_batch(model, interactions, k=10, item_features=None):
    n_users, n_items = interactions.shape
    results = []
    csr = interactions.tocsr()
    for u in range(n_users):
        true_mask = csr[u].toarray().ravel() > 0
        if not true_mask.any():
            results.append(0.0)
            continue
        scores = model.predict(np.repeat(u, n_items), np.arange(n_items), item_features=item_features)
        topk = np.argsort(-scores)[:k]
        hits = true_mask[topk].sum()
        results.append(hits / true_mask.sum())
    return np.array(results)

def coverage_at_n(model, hfm_inputs, N=10):
    item_map = hfm_inputs["item_map"]
    num_total_items = len(item_map)
    topn = get_top_n_recs_for_all_users(model, hfm_inputs, N=N)

    recommended_items = set()
    for uidx, top_items in topn.items():
        for iid in top_items:
            recommended_items.add(iid)
    coverage = len(recommended_items) / num_total_items if num_total_items > 0 else 0.0
    return coverage, len(recommended_items), num_total_items

def recommend_posts(user_id, model, hfm_inputs, interactions ,N=10):
    user_map = hfm_inputs["user_map"]
    item_map = hfm_inputs["item_map"]
    idx2post = hfm_inputs["idx2post"]
    item_features = hfm_inputs["item_features"]
    train_inter = hfm_inputs["train_interactions"]

    if user_id not in user_map:
        return []
    uidx = user_map[user_id]
    n_items = len(item_map)
    scores = model.predict(
        np.repeat(uidx, n_items),
        np.arange(n_items),
        item_features=item_features
    )
    seen_mask = train_inter.tocsr()[uidx].toarray().ravel() > 0
    scores[seen_mask] = -np.inf
    topk = np.argsort(-scores)[:N]
    test_ndcg = ndcg_at_k_batch(model, interactions, k=200, item_features=item_features).mean()

    return [(idx2post[i], float(scores[i])) for i in topk]

def ndcg_for_user(model, user_id, hfm_inputs, k=10):
    user_map = hfm_inputs["user_map"]
    test_inter = hfm_inputs["test_interactions"]
    item_features = hfm_inputs["item_features"]
    
    if user_id not in user_map:
        return None  # The user is not in the training/testing collection
    uidx = user_map[user_id]
    num_users, num_items = test_inter.shape
    csr_test = test_inter.tocsr()
    true_mask = csr_test[uidx].toarray().ravel() > 0
    if not true_mask.any():
        return None  # This user does not have ground truth in the test set

    # Score Prediction
    scores = model.predict(
        np.repeat(uidx, num_items),
        np.arange(num_items),
        item_features=item_features
    )
    topk = np.argsort(-scores)[:k]

    # Calculate DCG@k
    dcg = 0.0
    for rank, iid in enumerate(topk):
        if true_mask[iid]:
            dcg += 1.0 / log2(rank + 2)  

    ideal_n = min(k, true_mask.sum())
    idcg = sum(1.0 / log2(i + 2) for i in range(int(ideal_n)))
    if idcg == 0:
        return 0.0
    return dcg / idcg

def ndcg_for_user_all(model, user_id, hfm_inputs):

    user_map = hfm_inputs["user_map"]
    test_inter = hfm_inputs["test_interactions"]
    item_features = hfm_inputs["item_features"]
    
    if user_id not in user_map:
        return None  
    uidx = user_map[user_id]
    num_users, num_items = test_inter.shape
    csr_test = test_inter.tocsr()
    true_mask = csr_test[uidx].toarray().ravel() > 0
    if not true_mask.any():
        return None  
    
    scores = model.predict(
        np.repeat(uidx, num_items),
        np.arange(num_items),
        item_features=item_features
    )
    topk = np.argsort(-scores)

    dcg = 0.0
    for rank, iid in enumerate(topk):
        if true_mask[iid]:
            dcg += 1.0 / log2(rank + 2)  
    ideal_n = true_mask.sum()
    idcg = sum(1.0 / log2(i + 2) for i in range(int(ideal_n)))
    if idcg == 0:
        return 0.0
    return dcg / idcg


# ---- Main process ----
def Hybrid_Recommender():
    # ---- 0. Path and parameters ----
    votes_path = "votes2.csv"    # user_id, post_id, vote, created_utc
    posts_path = "posts2.csv"    # submission_id,title,selftext,num_comments,num_unique_commentators,ups,upvote_ratio,author,created_utc
    NUM_TOPICS = 10000
    TOP_K_AUTHORS = 500
    SEED = 42
    MIN_USER_INTERACTIONS = 40 
    MIN_ITEM_INTERACTIONS = 60

    # ---- 1. Read data ----
    reddit_data=pd.read_csv('./reddit_data.csv')
    reddit_data=reddit_data[reddit_data['text_only']==True]#SUBMISSION_ID
    data,_=get_dataframe()
    data_mul=reddit_data.merge(data,how='left',left_on='submission_id',right_on='SUBMISSION_ID')
    #data_sel_votes=data_mul[['USERNAME','SUBMISSION_ID','VOTE','created_utc']].drop_duplicates()
    posts_df=data_mul[['submission_id','title','selftext','num_comments','num_unique_commentators','ups','upvote_ratio','author','created_utc']].drop_duplicates()
    votes=data_mul[['USERNAME','SUBMISSION_ID','VOTE','created_utc']].drop_duplicates()
    votes['VOTE']=votes['VOTE'].replace({'upvote':1,'downvote':-1})
    votes = votes.rename(columns={
        'USERNAME': 'user_id',
        'SUBMISSION_ID': 'post_id',
        'VOTE': 'vote'
    })
    # Unified listing
    posts_df = posts_df.rename(columns={"submission_id": "post_id"})
    votes = votes.rename(columns={"post_id": "post_id", "user_id": "user_id"})

    # ---- 1.1 Only keep positive feedback ----
    votes = votes[votes["vote"] > 0].copy()

    # ---- 1.2 Filtering active users and active posts (long-tail processing) ----
    user_counts = votes.groupby("user_id").size()
    active_users = user_counts[user_counts >= MIN_USER_INTERACTIONS].index

    item_counts = votes.groupby("post_id").size()
    active_items = item_counts[item_counts >= MIN_ITEM_INTERACTIONS].index

    votes = votes[
        votes["user_id"].isin(active_users) &
        votes["post_id"].isin(active_items)
    ].copy()

    # Synchronous filtering posts_df retains only active items
    posts_df = posts_df[posts_df["post_id"].isin(active_items)].reset_index(drop=True)

    print(f"[filter] after active filtering: {votes['user_id'].nunique()} users, "
          f"{votes['post_id'].nunique()} items, {len(votes)} interactions")

    # ---- 2. Text merge + tokenization ----
    posts_df["title"] = posts_df["title"].fillna("")
    posts_df["selftext"] = posts_df["selftext"].fillna("")
    posts_df["full_text"] = posts_df["title"] + " . " + posts_df["selftext"]
    posts_df["tokens"] = posts_df["full_text"].map(simple_tokenize)

    # ---- 3. Thematic modeling (LDA) ----
    dictionary = corpora.Dictionary(posts_df["tokens"])
    dictionary.filter_extremes(no_below=5, no_above=0.5)
    bow_corpus = [dictionary.doc2bow(tok) for tok in posts_df["tokens"]]
    tfidf = TfidfModel(bow_corpus)
    tfidf_corpus = tfidf[bow_corpus]
    lda = LdaModel(
        tfidf_corpus,
        id2word=dictionary,
        num_topics=NUM_TOPICS,
        random_state=SEED,
        passes=10,
        alpha="auto",
        eta="auto"
    )
    posts_df["topic_vec"] = [
        get_topic_vector(lda, tfidf_corpus[i], NUM_TOPICS)
        for i in range(len(posts_df))
    ]

    # ---- 4. SBERT embedding---- 
    device = "mps" if hasattr(__import__("torch").backends, "mps") and __import__("torch").backends.mps.is_available() else "cpu"
    print(f"Using device for SBERT embeddings: {device}")
    sbert = SentenceTransformer("all-MiniLM-L6-v2", device=device)
    text_list = posts_df["full_text"].tolist()
    sbert_embs = sbert.encode(text_list, batch_size=32, show_progress_bar=True, normalize_embeddings=True)
    posts_df["sbert_emb"] = list(sbert_embs)

    # ---- 5. sentiment ----
    def extract_sentiment(text):
        tb = TextBlob(str(text))
        return tb.sentiment.polarity, tb.sentiment.subjectivity

    sentiments = posts_df["full_text"].map(extract_sentiment)
    posts_df["sentiment_polarity"] = sentiments.map(lambda x: x[0])
    posts_df["sentiment_subjectivity"] = sentiments.map(lambda x: x[1])

    # ---- 6. Other numerical characteristics & Normalization ----
    posts_df["created_dt"] = pd.to_datetime(posts_df["created_utc"], unit="s", utc=True)
    now = pd.Timestamp.utcnow()
    posts_df["age_hours"] = (now - posts_df["created_dt"]).dt.total_seconds() / 3600.0

    for col in ["num_comments", "num_unique_commentators", "ups", "age_hours"]:
        series = pd.to_numeric(posts_df[col], errors="coerce").fillna(0)
        posts_df[f"log_{col}"] = np.log1p(series)

    posts_df["word_count"] = posts_df["full_text"].map(lambda t: len(str(t).split()))
    posts_df["char_count"] = posts_df["full_text"].map(lambda t: len(str(t)))

    posts_df["log_word_count"] = np.log1p(posts_df["word_count"])
    posts_df["log_char_count"] = np.log1p(posts_df["char_count"])

    scaler = MinMaxScaler()
    to_scale = [
        "log_num_comments", "log_num_unique_commentators",
        "log_ups", "upvote_ratio", "log_age_hours",
        "log_word_count", "log_char_count"
    ]
    posts_df[to_scale] = scaler.fit_transform(posts_df[to_scale].fillna(0))

    # ---- 7. author bucket ----
    author_counts = posts_df["author"].value_counts()
    top_authors = set(author_counts.nlargest(TOP_K_AUTHORS).index)

    def bucket_author(a):
        return a if a in top_authors else "other"

    posts_df["author_bucket"] = posts_df["author"].map(bucket_author)

    # ---- 8.  item_feat_tuples ----
    item_feat_tuples = []
    for _, row in posts_df.iterrows():
        pid = row["post_id"]
        feat = {}

        for i, v in enumerate(row["topic_vec"]):
            feat[f"topic_{i}"] = float(v)
        for i, v in enumerate(row["sbert_emb"]):
            feat[f"sbert_{i}"] = float(v)
        feat["sentiment_polarity"] = float(row["sentiment_polarity"])
        feat["sentiment_subjectivity"] = float(row["sentiment_subjectivity"])
        feat["num_comments_norm"] = float(row["log_num_comments"])
        feat["num_unique_commentators_norm"] = float(row["log_num_unique_commentators"])
        feat["ups_norm"] = float(row["log_ups"])
        feat["upvote_ratio_norm"] = float(row["upvote_ratio"])
        feat["age_hours_norm"] = float(row["log_age_hours"])
        feat["word_count_norm"] = float(row["log_word_count"])
        feat["char_count_norm"] = float(row["log_char_count"])
        feat[f"author_{row['author_bucket']}"] = 1.0

        item_feat_tuples.append((pid, feat))

    # ---- 9. per-user holdout 拆分 ----
    train_df, val_df, test_df = per_user_holdout(votes, test_frac=0.2, val_frac=0.1, time_col='created_utc')

    # ---- 10. LightFM Dataset + interactions + item_features ----
    feat_name_set = set()
    for _, feat_dict in item_feat_tuples:
        feat_name_set.update(feat_dict.keys())
    feat_names = sorted(feat_name_set)

    dataset = Dataset()
    dataset.fit(
        users=votes["user_id"].unique(),
        items=posts_df["post_id"].unique(),
        item_features=feat_names
    )

    def build_inter(df):
        return dataset.build_interactions(
            [(row.user_id, row.post_id, 1.0) for row in df.itertuples()]
        )

    train_interactions, _ = build_inter(train_df)
    val_interactions, _ = build_inter(val_df)
    test_interactions, _ = build_inter(test_df)

    item_features = dataset.build_item_features(item_feat_tuples, normalize=False)
    user_map, id2user, item_map, id2item = dataset.mapping()
    idx2post = {idx: pid for pid, idx in item_map.items()}

    hfm_inputs = {
        "dataset": dataset,
        "train_interactions": train_interactions,
        "val_interactions": val_interactions,
        "test_interactions": test_interactions,
        "item_features": item_features,
        "user_map": user_map,
        "item_map": item_map,
        "idx2post": idx2post
    }

    # ---- 11. HyperPara grid search on validation ----
    param_grid = [
        {"no_components": 20, "loss": "warp",     "user_alpha": 1e-6, "item_alpha": 1e-6, "learning_rate": 0.05},
        {"no_components": 30, "loss": "warp-kos", "user_alpha": 1e-6, "item_alpha": 1e-6, "learning_rate": 0.05},
        {"no_components": 30, "loss": "bpr",      "user_alpha": 1e-7, "item_alpha": 1e-7, "learning_rate": 0.05},
    ]

    best_model = None
    best_cfg = None
    best_val_prec = -np.inf
    for cfg in param_grid:
        print(f"[grid] training with {cfg}")
        model = LightFM(
            no_components=cfg["no_components"],
            loss=cfg["loss"],
            user_alpha=cfg["user_alpha"],
            item_alpha=cfg["item_alpha"],
            learning_rate=cfg["learning_rate"],
            random_state=SEED
        )
        model.fit(
            train_interactions,
            item_features=item_features,
            epochs=25,
            num_threads=4,
            verbose=False
        )
        val_prec = precision_at_k(model, val_interactions, item_features=item_features, k=10).mean()
        val_ndcg = ndcg_at_k_batch(model, val_interactions, k=10, item_features=item_features).mean()
        print(f"  val Precision@10: {val_prec:.4f}, nDCG@10: {val_ndcg:.4f}")
        if val_prec > best_val_prec:
            best_val_prec = val_prec
            best_model = model
            best_cfg = cfg
    print(f"\nBest hyperparams: {best_cfg}, val Precision@10={best_val_prec:.4f}")

    # ---- 12. Final assessment ----
    model = best_model
    train_prec = precision_at_k(model, train_interactions, item_features=item_features, k=200).mean()
    test_prec = precision_at_k(model, test_interactions, item_features=item_features, k=10).mean()
    train_auc = auc_score(model, train_interactions, item_features=item_features).mean()
    test_auc = auc_score(model, test_interactions, item_features=item_features).mean()
    train_ndcg = ndcg_at_k_batch(model, train_interactions, k=200, item_features=item_features).mean()
    test_ndcg = ndcg_at_k_batch(model, test_interactions, k=200, item_features=item_features).mean()
    train_recall = recall_at_k_batch(model, train_interactions, k=200, item_features=item_features).mean()
    test_recall = recall_at_k_batch(model, test_interactions, k=200, item_features=item_features).mean()
    coverage, unique_rec_items, total_items = coverage_at_n(model, hfm_inputs, N=10)

    print("\n=== Final Evaluation ===")
    print(f"Train Precision@200: {train_prec:.4f}, Recall@200: {train_recall:.4f}, nDCG@200: {train_ndcg:.4f}")
    print(f" Test Precision@200: {test_prec:.4f}, Recall@200: {test_recall:.4f}, nDCG@200: {test_ndcg:.4f}")
    #print(f"Coverage@10: {coverage:.4f} ({unique_rec_items}/{total_items} distinct items recommended)")

    test_prec_per_user = precision_at_k(model, test_interactions, item_features=item_features, k=10)
    print(f"Test Precision@10 per-user median: {np.median(test_prec_per_user):.4f}, mean: {test_prec_per_user.mean():.4f}")

    # ---- 13. Recommend ----
    example_user = ['Raven2002', 'mguardian_north', 'Clen23', 'Reeses2150', 'spockspeare', 'apoeticturtle', 'Mash404', 'locks_are_paranoid', 'daygloviking', 'thx1138jr', 'Livelogikal', 'MingeyMackrel', 'uncertainusurper', 'lokier01', 'baddonkey', 'pierrekrahn', 'CubyChris', 'Adventurous_Guy', 'stratman42', 'VerbotenPublish']
    print(f"\nTop-5 recommendations for user {example_user}:")
    #recs = recommend_posts(example_user, model, hfm_inputs, N=5)
    # for pid, score in recs:
    #     print(f"  post {pid}, score={score:.4f}")
    for user in example_user:
        print(f'{user}: {ndcg_for_user(model,user,hfm_inputs,k=200)}')



   



if __name__ == "__main__":
    Hybrid_Recommender()
    

[nltk_data] Downloading package brown to /Users/hiking/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/hiking/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/hiking/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/hiking/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package conll2000 to
[nltk_data]     /Users/hiking/nltk_data...
[nltk_data]   Package conll2000 is already up-to-date!
[nltk_data] Downloading package movie_reviews to
[nltk_data]     /Users/hiking/nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!
/var/folders/j6/smjr9n410pgcjjpxcmckc9lh0000gn/T/ipykernel_10834/3569476229.py:243: DtypeWarning: Columns (5) have mi

[filter] after active filtering: 731 users, 332 items, 9803 interactions


/opt/anaconda3/envs/recommender_systems/lib/python3.11/site-packages/gensim/models/ldamodel.py:850: RuntimeWarning: overflow encountered in exp2
  perwordbound, np.exp2(-perwordbound), len(chunk), corpus_words


Using device for SBERT embeddings: mps


Batches:   0%|          | 0/11 [00:00<?, ?it/s]

[grid] training with {'no_components': 20, 'loss': 'warp', 'user_alpha': 1e-06, 'item_alpha': 1e-06, 'learning_rate': 0.05}
  val Precision@10: 0.0012, nDCG@10: 0.0026
[grid] training with {'no_components': 30, 'loss': 'warp-kos', 'user_alpha': 1e-06, 'item_alpha': 1e-06, 'learning_rate': 0.05}
  val Precision@10: 0.0000, nDCG@10: 0.0000
[grid] training with {'no_components': 30, 'loss': 'bpr', 'user_alpha': 1e-07, 'item_alpha': 1e-07, 'learning_rate': 0.05}
  val Precision@10: 0.0025, nDCG@10: 0.0043

Best hyperparams: {'no_components': 30, 'loss': 'bpr', 'user_alpha': 1e-07, 'item_alpha': 1e-07, 'learning_rate': 0.05}, val Precision@10=0.0025

=== Final Evaluation ===
Train Precision@200: 0.0520, Recall@200: 0.9993, nDCG@200: 0.7591
 Test Precision@200: 0.0039, Recall@200: 0.4926, nDCG@200: 0.1109
Test Precision@10 per-user median: 0.0000, mean: 0.0039

Top-5 recommendations for user ['Raven2002', 'mguardian_north', 'Clen23', 'Reeses2150', 'spockspeare', 'apoeticturtle', 'Mash404', '